# Workshop: Platform & Workspace First Steps

**Learning objective:** Attach a notebook to Serverless compute, navigate the Unity Catalog hierarchy, create and use a managed Volume, and read your first dataset file with Spark.

**Expected duration:** ~25 min


## Scenario

> *"It's your first day on the RetailHub data team. Before you can start building the analytics platform, you need to get comfortable with the workspace: attach this notebook to Serverless compute, explore your Unity Catalog environment, create a Volume for file landing, and run your first Spark reads."*


## Objectives

After completing this lab you will be able to:
- Attach a notebook to **Serverless** compute
- Navigate the Unity Catalog hierarchy (Catalog > Schema > Table/Volume)
- Create a managed Unity Catalog **Volume** with SQL and verify it via `information_schema`
- Write and read files in a Volume with `dbutils.fs`
- Read a dataset file from a Volume with `spark.read`


## Prerequisites

- Access to the Databricks workspace (URL + credentials provided by trainer)
- The trainer has run `00_pre_config.ipynb` — your catalog `retailhub_{your_name}` exists and the `datasets` Volume is populated


## Part 1: Attach to Serverless Compute (~5 min)

Serverless is the **primary compute** for this training — no cluster to create, nothing to configure, it starts in seconds.

1. Look at the **compute selector** in the top-right corner of this notebook
2. Click it and choose **Serverless**
3. Wait a few seconds — the dot turns green and the notebook is attached
4. Run the first code cell below (Part 3) to confirm everything works

> **Exam Tip:** Know the compute types: **Serverless** (instant start, Databricks-managed, default for notebooks/jobs/SQL), **All-Purpose clusters** (interactive, user-managed, minutes to start), and **Job clusters** (ephemeral, created per job run, cheaper DBU rate).


### 👨‍🏫 Instructor demo (classic compute) — you will use Serverless

> The trainer will briefly show how a **classic all-purpose cluster** is created — you do **not** need to create one. All labs in this training run on Serverless.

What the trainer shows (for recognition only):

| Setting | Typical value |
|---------|---------------|
| **Compute** > **Create compute** | entry point in the left sidebar |
| **Policy** | Personal Compute |
| **Access mode** | Single User (Dedicated) |
| **Databricks Runtime** | Latest LTS |
| **Terminate after** | 60 minutes of inactivity |

Why it matters: the exam still asks about all-purpose vs job clusters, runtimes, and auto-termination — even though serverless is the day-to-day default.


## Part 2: Explore Unity Catalog in the UI (~5 min)

1. In the left sidebar, click **Catalog**
2. Find your catalog: `retailhub_{your_name}` and expand it

```
retailhub_{your_name}          -- CATALOG (top-level namespace)
  ├── bronze                   -- SCHEMA (database)
  │     ├── (tables)           -- TABLES
  │     └── (volumes)          -- VOLUMES
  ├── silver                   -- SCHEMA
  ├── gold                     -- SCHEMA
  └── default                  -- SCHEMA
        └── datasets           -- VOLUME (managed, pre-populated)
```

3. Click the `default` schema > **Volumes** > `datasets` — browse the dataset files copied there for you
4. Note the tabs on a schema: **Tables**, **Volumes**, **Functions**

> **Exam Tip:** Unity Catalog uses a **3-level namespace**: `catalog.schema.object`. Always use fully qualified names in production code. **Managed volumes** store data in the metastore-managed location; **external volumes** point to existing cloud storage paths.


## Part 3: Hands-on Tasks (~20 min)

Now the executable part. Run the setup cell first — it validates your environment and exports the variables `CATALOG`, `BRONZE_SCHEMA`, `SILVER_SCHEMA`, `GOLD_SCHEMA`, and `DATASET_PATH` used in every task.


In [ ]:
%run ../setup/00_setup

## Task 1: Verify Your Catalog & Schemas

Confirm with SQL that your session points at the right catalog and that the medallion schemas exist.

**What you need to do:** Use `spark.sql()` to select `current_catalog()`, then list the schemas in your catalog with `SHOW SCHEMAS`.


**Guidance — Task 01**

The goal is to verify with code what you just saw in the Catalog Explorer UI.

**Context functions**
`current_catalog()` and `current_schema()` return the active context of your session. The setup notebook already ran `USE CATALOG` for you, so `current_catalog()` should match the `CATALOG` variable.

**Listing schemas**
`SHOW SCHEMAS IN <catalog>` returns one row per schema. In Python you can collect the rows and build a list:

```python
rows = spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()
names = [r[0] for r in rows]
```

**Things to think about**
- Why is it safer to reference `{CATALOG}` from the setup variable than to hardcode the catalog name?
- What is the difference between `SHOW SCHEMAS` and querying `information_schema.schemata`?


In [ ]:
current_cat = spark.sql("SELECT current_catalog()").first()[0]

schema_names = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]

print(f"Current catalog: {current_cat}")
print(f"Schemas:         {schema_names}")

In [ ]:
# -- Validation --
assert current_cat == CATALOG, f"Expected current catalog '{CATALOG}', got '{current_cat}'"
for s in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    assert s in schema_names, f"Schema '{s}' is missing from {schema_names}"
print(f"Task 1 OK: catalog '{CATALOG}' active, schemas {BRONZE_SCHEMA}/{SILVER_SCHEMA}/{GOLD_SCHEMA} present")

## Task 2: Create a Managed Volume with SQL

In Part 2 you saw the pre-created `datasets` Volume. Now create your **own managed Volume** named `landing` in the **bronze** schema — a landing zone for raw files.

**What you need to do:** Run a `CREATE VOLUME IF NOT EXISTS` statement, then verify the Volume exists by querying `information_schema.volumes`.


**Guidance — Task 02**

The goal is to create a managed Volume with SQL instead of the UI.

**CREATE VOLUME syntax**

```sql
CREATE VOLUME IF NOT EXISTS catalog.schema.volume_name
COMMENT 'optional description'
```

A **managed** Volume needs no `LOCATION` clause — Unity Catalog stores its files in the schema's managed storage location. (`CREATE EXTERNAL VOLUME ... LOCATION 'abfss://...'` would create an external one.)

**Verifying via information_schema**
Every catalog exposes a SQL-standard `information_schema`. Volumes appear in `<catalog>.information_schema.volumes` with columns such as `volume_catalog`, `volume_schema`, `volume_name`, `volume_type`.

**Things to think about**
- Where do the files of a managed Volume physically live? Who controls that location?
- What happens to the files when you `DROP VOLUME` a managed vs an external Volume?


In [ ]:
LANDING_VOLUME = "landing"

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{LANDING_VOLUME}
    COMMENT 'Landing zone for raw files (lab 01)'
""")

volumes_df = spark.sql(f"""
    SELECT volume_catalog, volume_schema, volume_name, volume_type
    FROM {CATALOG}.information_schema.volumes
    WHERE volume_schema = '{BRONZE_SCHEMA}'
      AND volume_name = '{LANDING_VOLUME}'
""")

display(volumes_df)

In [ ]:
# -- Validation --
vol_rows = volumes_df.collect()
assert len(vol_rows) == 1, f"Expected exactly 1 matching volume, got {len(vol_rows)}"
row = vol_rows[0].asDict()
assert row.get("volume_name") == LANDING_VOLUME, f"Unexpected volume row: {row}"
assert row.get("volume_type") == "MANAGED", f"Volume should be MANAGED, got {row.get('volume_type')}"
print(f"Task 2 OK: managed volume {CATALOG}.{BRONZE_SCHEMA}.{LANDING_VOLUME} exists")

## Task 3: Write and Read a File with dbutils.fs

Volumes are accessed through the POSIX-style path `/Volumes/<catalog>/<schema>/<volume>/...`. Write a small text file into your new `landing` Volume and read it back.

**What you need to do:**
1. Build the volume path: `/Volumes/{CATALOG}/{BRONZE_SCHEMA}/landing`
2. Write a file `hello_retailhub.txt` with `dbutils.fs.put()`
3. Read it back with `dbutils.fs.head()` and list the volume with `dbutils.fs.ls()`


**Guidance — Task 03**

The goal is to use the `dbutils.fs` file-system utilities against a Unity Catalog Volume.

**The dbutils.fs trio**

| Call | Does |
|------|------|
| `dbutils.fs.put(path, contents, True)` | Write a string to a file (`True` = overwrite) |
| `dbutils.fs.head(path)` | Return the first bytes of a file as a string |
| `dbutils.fs.ls(path)` | List directory entries (`FileInfo` objects with `.name`, `.size`) |

**Volume paths**
A Volume is addressed as `/Volumes/<catalog>/<schema>/<volume>` — the same path works in `dbutils.fs`, `spark.read`, SQL `read_files()`, and `%sh` (with a `/Volumes/...` mount view).

**Things to think about**
- How is governance different when writing to a Volume vs writing to a raw cloud-storage URI?
- Which privileges does a user need on the Volume to run `dbutils.fs.put()` on it? (`READ VOLUME` / `WRITE VOLUME`)


In [ ]:
LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{LANDING_VOLUME}"
FILE_PATH    = f"{LANDING_PATH}/hello_retailhub.txt"
MESSAGE      = f"Hello from {CATALOG} — RetailHub training!"

dbutils.fs.put(FILE_PATH, MESSAGE, True)

content = dbutils.fs.head(FILE_PATH)

# List the volume contents
for f in dbutils.fs.ls(LANDING_PATH):
    print(f"{f.name}  ({f.size} bytes)")
print(f"File content: {content}")

In [ ]:
# -- Validation --
assert content == MESSAGE, f"Round-trip mismatch: expected '{MESSAGE}', got '{content}'"
file_names = [f.name for f in dbutils.fs.ls(LANDING_PATH)]
assert "hello_retailhub.txt" in file_names, f"File not found in volume: {file_names}"
print(f"Task 3 OK: file written and read back from {FILE_PATH}")

## Task 4: Read a Dataset File with Spark

Time for real data. The `datasets` Volume (exported as `DATASET_PATH` by the setup) holds the RetailHub files. Read the customers CSV into a DataFrame.

**What you need to do:** Read `{DATASET_PATH}/customers/customers.csv` with `spark.read.format("csv")`, `header=true` and `inferSchema=true`, then inspect the schema.


**Guidance — Task 04**

The goal is your first Spark read — the pattern you will use throughout the training.

**Reader pattern**

```python
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(path))
```

`spark.read` is lazy — nothing happens until an **action** such as `display()`, `.count()` or `.show()` runs.

**Inspecting the result**
- `df.printSchema()` — column names and inferred types
- `df.columns` — list of column names
- `df.count()` — row count (an action — triggers the actual read)

**Things to think about**
- Which types did Spark infer for each column — are they what you expected?
- In LAB 02 you will replace `inferSchema` with an explicit `StructType`. Why is that better in production?


In [ ]:
CUSTOMERS_CSV = f"{DATASET_PATH}/customers/customers.csv"

customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CUSTOMERS_CSV)
)

customers_df.printSchema()
display(customers_df.limit(5))

In [ ]:
# -- Validation --
customer_count = customers_df.count()
assert customer_count > 0, "customers_df should not be empty"
assert "customer_id" in customers_df.columns, f"Expected 'customer_id' column, got {customers_df.columns}"
print(f"Task 4 OK: {customer_count} customers read from {CUSTOMERS_CSV}")

## Summary

In this lab you:
- Attached this notebook to **Serverless** compute (classic clusters: instructor demo only)
- Explored the Unity Catalog 3-level namespace in the UI
- Created a **managed Volume** with SQL and verified it via `information_schema.volumes`
- Wrote and read a file in the Volume with `dbutils.fs`
- Read your first RetailHub dataset file with `spark.read`

> **What's next:** In LAB 02 you will use these files to build your first ELT ingestion pipeline — reading CSV/JSON/Parquet with explicit schemas and writing Bronze Delta tables.


## Cleanup

In [ ]:
# Remove the lab artifacts (the pre-populated `datasets` volume is NOT touched)
dbutils.fs.rm(FILE_PATH)
spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.{LANDING_VOLUME}")
print("Lab cleanup complete")

← [01 — Platform & Workspace](../demo/01_platform_and_workspace.ipynb) | **[ README](../../../README.md)** | [02 — Data Ingestion →](../demo/02_data_ingestion.ipynb)